#### WHat are the Imports
1. pdf loader from langchain
2. to create chunks - RecursiveCharacterTextSplitter. ( comes with overlap in between chunking so not to miss text )
3. from langchain_openai import ChatOpenAI - just like we used connection client to call open ai to send and recieve response, its a wrappen built on top of that, does same thing without writing the code.

In [17]:
from langchain_openai import ChatOpenAI
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from dotenv import load_dotenv
load_dotenv()

from langchain_openai import OpenAIEmbeddings

#### Step 0 : Load pdf into text formt

In [ ]:
# text_data = PyPDFLoader("NovaS.pdf").load()
# text_data

# text_content = []

# for i in text_data:
#     text_content.append(i.page_content)

# print(text_content)  # Print the first 1000 characters of the text content
text_data = PyPDFLoader("NovaS.pdf").load()

# Changing Metadata of the document
for page in text_data:
    page.metadata["source"] = "NovaS.pdf"

text_data

list

#### Step 2 : Splitting the text into chunks for Embeddings

In [20]:
splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)

chunks = splitter.create_documents(text_content)
len(chunks)

101

#### Step 3 : Create Embeddings of the chunks and storing into Vector Database
1. We import the vector db
2. We have to select an embedding model from openai

In [21]:
from langchain_community.vectorstores import Chroma

embed_model = OpenAIEmbeddings(model="text-embedding-3-small")
chroma_db = Chroma.from_documents(chunks, embed_model, persist_directory="./chroma_db")

#### Step 4 : Connection and Retrieval

In [22]:
chroma_db_con = Chroma(persist_directory="./chroma_db", embedding_function=embed_model)

C:\Users\swapn\AppData\Local\Temp\ipykernel_21556\3816121341.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  chroma_db_con = Chroma(persist_directory="./chroma_db", embedding_function=embed_model)


In [23]:
chroma_db_con.similarity_search("By 2019, how many employees were there?", k=3)

[Document(metadata={}, page_content='By the beginning of 2019, the organization had grown to more than fifteen employees. This'),
 Document(metadata={}, page_content='During 2023, the organization experienced steady growth. The number of employees'),
 Document(metadata={}, page_content='number of employees increased again, and the company also started hiring people who')]

#### Step 5 : LLM Integration and Answer GEneration

In [24]:
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

In [26]:
user_query = input("Enter your question: ")

rel_chunks = chroma_db_con.similarity_search(user_query, k=3)

rel_chunks_content = []
for i, chunk in enumerate(rel_chunks):
    rel_chunks_content.append(chunk.page_content)
rel_chunks_content = str(rel_chunks_content)

llm.invoke(f"{user_query}, Use the following context to answer the question: {rel_chunks_content}")

AIMessage(content='By 2019, the organization had more than fifteen employees.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 81, 'total_tokens': 94, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-EQBOw3w0G5EspdlqIyo46oiHSdvYb', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0bee4-b95e-70a1-a2f5-448b508efc17-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 81, 'output_tokens': 13, 'total_tokens': 94, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0